In [3]:
# Permite importar src/ desde la carpeta notebooks/
import sys
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import DIR_DATA_RAW

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")

In [ ]:
from src.faenas import cargar_faenas, construir_viajes, unir_solapados
from src.telemetria import asignar_viaje, cruzar_nombres

viajes, calas, obs = construir_viajes(cargar_faenas(DIR_DATA_RAW / "faenas" / "pod.csv"))
viajes, calas = unir_solapados(viajes, calas)
display(obs)

tel = pd.concat([pd.read_parquet(a) for a in (DIR_DATA_RAW / "telemetria").glob("*.parquet")])
tel["fecha"] = pd.to_datetime(tel["fecha"])

# 1. ¿Los nombres coinciden entre telemetría y faenas?
nombres = cruzar_nombres(tel, viajes)
display(nombres[~(nombres.en_telemetria & nombres.en_faenas)])

# 2. Asignar viaje
tel_v = asignar_viaje(tel, viajes)
print(f"Registros con viaje: {tel_v['IdViaje'].notna().mean():.1%}")
display(tel_v.groupby("ep")["IdViaje"].apply(lambda s: s.notna().mean()).sort_values().to_frame("% con viaje"))

# 3. ¿Cada cuánto llega un registro de telemetría?
print(tel_v.groupby("ep")["fecha"].diff().dt.total_seconds().div(60).describe(percentiles=[.5, .9, .99]))

In [ ]:
# 1. Quitar duplicados (misma embarcación y misma hora)
tel["ep"] = tel["ep"].astype(str).str.strip().str.upper()
antes = len(tel)
tel = tel.drop_duplicates(subset=["ep", "fecha"]).sort_values(["ep", "fecha"])
print(f"Duplicados eliminados: {antes - len(tel):,}")

# 2. % con viaje SOLO dentro de los meses que tienen faenas
tel_v = asignar_viaje(tel, viajes)
meses_faenas = set(viajes["ZarpeCorregido"].dt.to_period("M"))
dentro = tel_v["fecha"].dt.to_period("M").isin(meses_faenas)
print(f"Registros fuera del periodo de faenas: {(~dentro).mean():.1%}")
print(f"% con viaje (dentro del periodo): {tel_v.loc[dentro, 'IdViaje'].notna().mean():.1%}")

# 3. Velocidad y RPM alrededor de cada inicio de cala (en bloques de 15 min)
c = calas.merge(viajes[["IdViaje", "Embarcacion"]], on="IdViaje")
c = c[c["Embarcacion"].isin(tel["ep"].unique())]

filas = []
for ep, grupo in c.groupby("Embarcacion"):
    t = tel[tel["ep"] == ep][["fecha", "Velocidad", "rpm", "gal_h"]]
    for inicio in grupo["InicioCala"]:
        w = t[t["fecha"].between(inicio - pd.Timedelta(hours=6), inicio + pd.Timedelta(hours=6))].copy()
        w["min_rel"] = ((w["fecha"] - inicio).dt.total_seconds() / 60 / 15).round() * 15
        filas.append(w)

perfil = pd.concat(filas).groupby("min_rel")[["Velocidad", "rpm", "gal_h"]].median()

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, col in zip(axes, perfil.columns):
    ax.plot(perfil.index, perfil[col], marker="o", ms=3)
    ax.axvline(0, color="red", ls="--", label="InicioCala")
    ax.set_ylabel(col)
axes[0].legend()
axes[-1].set_xlabel("minutos respecto al inicio de cala")
plt.tight_layout()

display(perfil.loc[-120:240])

In [ ]:
from src.calas import estimar_ventana_calas

v_ok = viajes[viajes["FuenteFin"] == "real"]
ventanas = estimar_ventana_calas(tel, calas[calas["IdViaje"].isin(v_ok["IdViaje"])], v_ok)

print(f"Calas detectadas: {ventanas['CalaDetectada'].mean():.1%} de {len(ventanas):,}")
display(ventanas[["DuracionEstMin", "DesfaseInicioMin", "DesfaseFinMin"]]
        .describe(percentiles=[.05, .25, .5, .75, .95]))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["DuracionEstMin", "DesfaseInicioMin", "DesfaseFinMin"]):
    ventanas[col].dropna().clip(-240, 240).hist(bins=32, ax=ax)
    ax.set_title(col)
plt.tight_layout()

In [ ]:
from src.etiquetas import ventanas_finales, etiquetar

vf = ventanas_finales(ventanas)
display(vf["MetodoVentana"].value_counts(normalize=True).to_frame("%"))

tel_e = etiquetar(tel_v[tel_v["IdViaje"].isna() | tel_v["IdViaje"].isin(v_ok["IdViaje"])], vf)
display(tel_e["Etiqueta"].value_counts().to_frame("registros"))
display(tel_e.groupby("Etiqueta")[["Velocidad", "rpm", "gal_h"]].median())

# Ver 3 viajes al azar: velocidad en el tiempo, coloreada por etiqueta
colores = {"ida": "tab:blue", "busqueda": "tab:orange", "cala": "tab:red", "retorno": "tab:green"}
for idv in tel_e.loc[tel_e["Etiqueta"] == "cala", "IdViaje"].drop_duplicates().sample(3, random_state=7):
    d = tel_e[tel_e["IdViaje"] == idv]
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(d["fecha"], d["Velocidad"], color="lightgray", zorder=1)
    for etq, g in d.groupby("Etiqueta"):
        ax.scatter(g["fecha"], g["Velocidad"], s=18, label=etq, color=colores.get(etq, "gray"), zorder=2)
    for _, c in calas[calas["IdViaje"] == idv].iterrows():
        ax.axvline(c["InicioCala"], color="red", ls=":", alpha=0.6)
    ax.set_title(f"Viaje {idv} – {d['ep'].iloc[0]}")
    ax.legend(loc="upper right", ncol=4)
    plt.tight_layout()

In [ ]:
from src.rumbo import verificar_codigos_rumbo, agregar_variables_rumbo

display(verificar_codigos_rumbo(tel_v))

In [ ]:
from src.rumbo import rumbo_gps

t = tel_v.sort_values(["ep", "fecha"]).copy()
t["tipo_rumbo"] = t["Rumbo"].map(lambda x: type(x).__name__)
g = t.groupby("ep")
lat0, lon0 = np.radians(g["Latitud"].shift()), np.radians(g["Longitud"].shift())
lat1, lon1 = np.radians(t["Latitud"]), np.radians(t["Longitud"])
a = np.sin((lat1 - lat0) / 2)**2 + np.cos(lat0) * np.cos(lat1) * np.sin((lon1 - lon0) / 2)**2
t["dist_km"] = 6371 * 2 * np.arcsin(np.sqrt(a))
t["dt_h"] = g["fecha"].diff().dt.total_seconds() / 3600
t["vel_gps"] = t["dist_km"] / 1.852 / t["dt_h"]          # nudos
t["misma_pos"] = t["dist_km"] < 0.01                      # no se movió (<10 m)

ok = t["dt_h"].between(0.1, 0.5) & (t["Velocidad"] >= 6)  # navegando, registros consecutivos
resumen = t[ok].groupby(["origen", "tipo_rumbo"]).agg(
    n=("Velocidad", "size"),
    vel_reportada=("Velocidad", "median"),
    vel_gps=("vel_gps", "median"),
    pct_misma_posicion=("misma_pos", "mean"),
)
display(resumen.round(2))

# Ejemplo crudo de una fuente histórica: ¿cómo lucen las coordenadas?
display(t[t["origen"] == "mp_m_h"][["ep", "fecha", "Latitud", "Longitud", "Velocidad", "Rumbo", "dist_km"]].head(15))